# Session 11: Exercises

**Before session 12.** Predict, then measure. Answers go in the **✍️** cells, and code TODOs are
marked `# TODO`.

Exercises 1 and 5 make **no** model calls. The others say roughly how many they make. Every policy
search and every memory also embeds with `gemini-embedding-2`, so you need `GEMINI_API_KEY` in
`labs/.env`.

Like the lesson notebook, this one keeps its own save files (`labs/.state/s11-exercises-*`), deleted
at the start, so your `chat.py` tickets are never touched.

## Setup: given, just run it

In [ ]:
import json
import os
import sys
import uuid
import warnings
from pathlib import Path
from typing import TypedDict

from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", category=LangChainBetaWarning)
warnings.filterwarnings("ignore", message="IProgress not found")

HERE = Path.cwd()
STATE = HERE.parent / ".state"
for name, file in {"SHOPWISE_CHECKPOINT_DB": "s11-exercises-checkpoints.sqlite",
                   "SHOPWISE_MEMORY_DB": "s11-exercises-memory.sqlite",
                   "SHOPWISE_REFUND_LEDGER": "s11-exercises-refunds.jsonl"}.items():
    (STATE / file).unlink(missing_ok=True)
    os.environ[name] = str(STATE / file)
sys.path.insert(0, str(HERE / "project"))

from langgraph.checkpoint.memory import InMemorySaver     # noqa: E402
from langgraph.graph import END, START, StateGraph        # noqa: E402
from langgraph.types import Command, interrupt            # noqa: E402

import shopwise_data                                      # noqa: E402
import workflows.inbox.graph as wiring                    # noqa: E402
import workflows.inbox.nodes as nodes                     # noqa: E402
from config import settings                               # noqa: E402
from inbox import load_all_tickets                        # noqa: E402
from middleware import context_for                        # noqa: E402
from workflows.events import paused_on, run_config, stream_ticket   # noqa: E402
from workflows.inbox import open_inbox                    # noqa: E402

TICKETS = load_all_tickets()
_inbox = open_inbox()
graph = _inbox.__enter__()


class Toy(TypedDict, total=False):
    order: str
    decision: str
    log: list


def one_node(fn):
    """A one-node graph with a fresh checkpointer. Returns (graph, config)."""
    builder = StateGraph(Toy)
    builder.add_node("node", fn)
    builder.add_edge(START, "node")
    builder.add_edge("node", END)
    return builder.compile(checkpointer=InMemorySaver()), {"configurable": {"thread_id": str(uuid.uuid4())}}


def events(thread, turn, ctx):
    """Run one turn; return its events as a list (tokens included)."""
    return list(stream_ticket(graph, thread, turn, ctx))


print(f"{settings.APP_NAME} v{settings.VERSION} · {settings.model_id}")

## ⭐ Exercise 1: The double print

**What this teaches:** rule 1. On resume the node runs again from its first line.

The node below prints once before its `interrupt()`. Run it as given: pause, then resume. Then move the
`print` to **after** the `interrupt()` and run it again. No model calls.

In [ ]:
def billing_step(state):
    print("   checking eligibility…")              # TODO: move this line below the interrupt
    decision = interrupt({"order_id": state["order"], "amount": 129.0})
    return {"decision": decision}


toy, cfg = one_node(billing_step)
print("first run:")
toy.invoke({"order": "ORD-5001"}, cfg)
print("resume:")
print("  ", toy.invoke(Command(resume="approve"), cfg))

**✍️ Your answer.** In two sentences: why does the line print twice above the interrupt and once below it, and why is the rule "interrupt early"?

## ⭐⭐ Exercise 2: Reject with a reason

**What this teaches:** the reject path, at the raw `interrupt()` level (session 7 did it through the
middleware).

Run TCK-M05 to its pause (about 4 calls). Then resume it as a **manager** with
`{"type": "reject", "reason": "..."}` (about 2 calls). Then prove, from the saved state and the ledger,
not from the reply's wording, that:

1. the customer is told the reason;
2. `issue_refund` never ran: the ledger is empty, and REF-9001 is still `requested`.

In [ ]:
ticket = TICKETS["TCK-M05"]
first = events("TCK-M05", ticket, context_for(ticket["from"]))
print("paused:", [e.payload["order_id"] for e in first if e.type == "interrupt"])

# TODO: resume as a manager with a reject and a reason
# TODO: print the reply, the saved `decisions` and `refunds`, the ledger, and REF-9001's status

**✍️ Your answer.** Which line of evidence proves `issue_refund` never ran, and why is the reply's wording not proof?

## ⭐⭐ Exercise 3: Fork the misroute

**What this teaches:** time travel as a debugging tool.

Run TCK-M09 (about 4 calls; it may pause on a refund). Find the checkpoint where `split` chose the
category, fork it with the right one (a warranty question for the policy desk), and run the fork
forward (about 5 calls). Print the corrected reply.

In [ ]:
ticket = TICKETS["TCK-M09"]
events("TCK-M09", ticket, context_for(ticket["from"]))
print("split chose:", [s.category for s in graph.get_state(run_config("TCK-M09")).values["sub_issues"]])

# TODO: walk graph.get_state_history(...) to the checkpoint whose .next lists the desks
# TODO: graph.update_state(that.config, {"sub_issues": [...]}, as_node="split")
# TODO: graph.invoke(None, fork_config, context=...) and print the reply

**✍️ Your answer.** Two sentences: what is this for in production?

## ⭐⭐ Exercise 4: A new progress event

**What this teaches:** nodes talk while they work; the event vocabulary is a contract you can check.

1. Write a replacement `orders_desk(task, runtime)` that emits a progress event *"Looking up your
   order…"* (use `nodes.progress`) and then calls `nodes.run_desk("orders", task, runtime)`.
2. Put it in `wiring.DESK_NODES["orders"]`, rebuild the graph with the same checkpointer and store,
   and run *"Where is my watch, ORD-5003?"* as Tanvir (about 5 calls). The progress line must appear
   before the reply.
3. A check, as an `assert`: every `token` event came from `compose`. The reply is the only text the
   customer sees streamed, so **the tokens joined together must equal the `done` event's reply**
   (compose removes stray non-citation brackets after streaming, so compare after applying
   `nodes.NOT_A_CITATION`).

In [ ]:
# TODO 1: def orders_desk(task, runtime): ...
# TODO 2: wiring.DESK_NODES["orders"] = orders_desk; rebuild with
#         wiring.build_inbox(checkpointer=graph.checkpointer, store=graph.store)
# TODO 3: run one turn, print progress and tool events in order, then assert the tokens match the reply

wiring.DESK_NODES["orders"] = nodes.orders_desk          # put v8 back when you are done

**✍️ Your answer.** Why is the check "tokens == reply" rather than "no token mentions JSON"?

## ⭐⭐⭐ Exercise 5: Spot the bug

**What this teaches:** the four rules, applied to code you did not write. No model calls.

Each snippet breaks one rule. For each: name the rule, explain what goes wrong in one sentence, fix it,
and run both versions to prove it.

**A.** A refund record is created, then the manager is asked:
```python
def refund_step(state):
    LEDGER.append({"order": state["order"], "status": "pending"})
    decision = interrupt({"order": state["order"]})
    ...
```
**B.** A careful developer wraps everything:
```python
def refund_step(state):
    try:
        decision = interrupt({"order": state["order"]})
        ...
    except Exception:
        decision = "error"
```
**C.** Ask about every open refund, read fresh from the database each time:
```python
def ask_each(state):
    for order in open_refunds_from_db():
        answers[order] = interrupt(f"refund {order}?")
```

In [ ]:
LEDGER = []
OPEN = ["ORD-5001", "ORD-5004"]

# TODO A: run the broken version (pause, resume) and print LEDGER; then a fixed version
# TODO B: run the broken version and show it never pauses; then a fixed version
# TODO C: pause, change OPEN, resume, and show the answer landing on the wrong order; then a fixed version

**✍️ Your answer.** Map each snippet to its rule.

## ⭐⭐⭐ Exercise 6: When not to rewrite

**What this teaches:** a technique has to earn its call, including on the questions that never needed it.

`rewrite_query` makes no call when there is no history: the "skip" guard. Measure what the guard saves,
and whether removing it would hurt. On the gold set's **answerable** questions, asked cold (no history),
score two arms on the policy graph's straight shape (about 30 calls each):

- **guard**: `rewrite_query([], q)` (returns `q` unchanged, no call), then the policy graph;
- **no guard**: call the standalone prompt anyway, with history `"(none)"`, then the policy graph.

Report retrieval, citation and calls per question for both, list every question the unguarded rewrite
changed, and write the `kb/RAG-DECISIONS.md` row.

In [ ]:
from evals import run_eval                            # noqa: E402
from kb.query import STANDALONE_PROMPT, rewrite_query  # noqa: E402

# TODO: build two targets (question -> {"answer", "documents", "calls", "asked"}), score both with
#       run_eval.score_row / run_eval.summarise, print run_eval.table(...), and list the changed questions

**✍️ Your answer.** Your `RAG-DECISIONS.md` row, and the decision.

In [ ]:
_inbox.__exit__(None, None, None)                 # close the SQLite connections